# Personalized news POC: GPT web search + BM25/semantic retrieval

This experiment keeps the CRM preparation and separate outreach-priority logic from `personalized_news_poc.ipynb`, but replaces Tavily with the OpenAI Responses API `web_search` tool through Portkey. It makes one shared initial web-search call per normalized interest, even when many contacts share that interest. If that response contains duplicate or non-direct URLs, one shared cached repair call fills the missing candidates. Canonical URLs are deduplicated, and BM25 plus embedding similarity select up to seven unique results. There is no reranker.

Only each article's title, summary, and source URL are retained from web search. All generated files are written beneath `gpt_web_search_output/`.

In [ ]:
%pip install -q pandas numpy scikit-learn openpyxl openai langchain-openai python-dotenv truststore

In [ ]:
from __future__ import annotations

import hashlib
import importlib
import json
import math
import os
import re
from collections import Counter
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urlparse, urlunparse

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv('.env', override=True)

import portkey_profiles
import embedding
import llm_service

portkey_profiles = importlib.reload(portkey_profiles)
embedding = importlib.reload(embedding)
llm_service = importlib.reload(llm_service)

EmbeddingClient = embedding.EmbeddingClient
get_embedding_model = embedding.get_embedding_model
get_llm_model = llm_service.get_llm_model
search_news_async = llm_service.search_news_async
WebSearchTimeoutError = llm_service.WebSearchTimeoutError

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_columns', 30)

INPUT_XLSX = Path('CRM_Input_template.xlsx')
OUTPUT_DIR = Path('gpt_web_search_output')
CACHE_DIR = OUTPUT_DIR / 'cache'
OUTPUT_XLSX = OUTPUT_DIR / 'personalized_news_gpt_web_search.xlsx'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

LOOKBACK_DAYS = 30
SEARCH_RESULTS_PER_INTEREST = 10
TOP_RESULTS_PER_INTEREST = 7
BM25_K1 = 1.5
BM25_B = 0.75
SEMANTIC_WEIGHT = 0.60
BM25_WEIGHT = 0.40
REFRESH_NEWS = False
EMBEDDING_MODEL = get_embedding_model()

# Local estimate only; official OpenAI Standard short-context rates as of 2026-10-07.
LUNA_INPUT_USD_PER_MILLION = 0.10
LUNA_CACHED_INPUT_USD_PER_MILLION = 0.01
LUNA_CACHE_WRITE_USD_PER_MILLION = 0.125
LUNA_OUTPUT_USD_PER_MILLION = 0.50
WEB_SEARCH_USD_PER_CALL = 0.01
PRICING_AS_OF = '2026-10-07'

assert math.isclose(SEMANTIC_WEIGHT + BM25_WEIGHT, 1.0)
print('Outputs:', OUTPUT_DIR.resolve())
print('Embedding model:', EMBEDDING_MODEL)
print('Web-search model:', os.getenv('PORTKEY_LLM_MODEL') or 'NOT SET')

## 1. Load the minimal CRM and calculate outreach priority separately

Article ranking uses interests only. Seniority, tier, relationship quality, and engagement timing remain in a separate operational score and never affect retrieval.

In [ ]:
def clean_text(value) -> str:
    if pd.isna(value):
        return ''
    return ' '.join(str(value).split())


contact_columns = [
    'Company Name', 'Contact Person Full Name', 'Designation', 'Seniority',
    'Relationship Quality', 'Automatic tier assigned',
    'Manual Tier change?', 'Professional interests Interests',
    'Personal Interests', 'Last engagement date',
]
account_columns = [
    'Account Name', 'Sector', 'Products/ Services', 'Competitors', 'Headquarters',
]

contacts_raw = pd.read_excel(
    INPUT_XLSX, sheet_name='Input Contact Information',
    usecols=contact_columns, engine='openpyxl',
)
accounts_raw = pd.read_excel(
    INPUT_XLSX, sheet_name='Accounts Overview',
    usecols=account_columns, engine='openpyxl',
)

contacts = contacts_raw.rename(columns={
    'Company Name': 'company',
    'Contact Person Full Name': 'contact',
    'Designation': 'designation',
    'Seniority': 'seniority',
    'Relationship Quality': 'relationship_quality',
    'Automatic tier assigned': 'automatic_tier',
    'Manual Tier change?': 'manual_tier',
    'Professional interests Interests': 'professional_interests',
    'Personal Interests': 'personal_interests',
    'Last engagement date': 'last_engagement_date',
})
accounts = accounts_raw.rename(columns={
    'Account Name': 'company', 'Sector': 'sector',
    'Products/ Services': 'products_services',
    'Competitors': 'competitors', 'Headquarters': 'headquarters',
})

contacts = contacts[contacts['contact'].notna() & contacts['company'].notna()].copy()
contacts = contacts.drop_duplicates(['contact', 'company'], keep='first')
accounts = accounts[accounts['company'].notna()].drop_duplicates('company', keep='first').copy()
for column in contacts.columns.drop('last_engagement_date'):
    contacts[column] = contacts[column].map(clean_text)
for column in accounts.columns:
    accounts[column] = accounts[column].map(clean_text)
contacts['last_engagement_date'] = pd.to_datetime(
    contacts['last_engagement_date'], errors='coerce'
)
contacts['tier'] = contacts['manual_tier'].where(
    contacts['manual_tier'].ne(''), contacts['automatic_tier']
)

crm = contacts.merge(accounts, on='company', how='left', validate='many_to_one')
missing_accounts = crm.loc[crm['sector'].eq('') | crm['sector'].isna(), 'company'].unique()
assert len(missing_accounts) == 0, f'Missing account context for: {missing_accounts}'
crm['contact_id'] = [f'C{i:04d}' for i in range(1, len(crm) + 1)]
print(f'{len(crm)} contacts across {crm.company.nunique()} accounts')
display(crm[['contact_id', 'contact', 'company', 'professional_interests', 'personal_interests']].head())

In [ ]:
LEVEL_SCORE = {'High': 1.0, 'Medium': 0.6, 'Low': 0.2}
TIER_SCORE = {'Tier 1': 1.0, 'Tier 2': 0.6, 'Tier 3': 0.2}
CADENCE_DAYS = {'Tier 1': 30, 'Tier 2': 90, 'Tier 3': 180}
REVIEW_THRESHOLD_DAYS = {'Tier 1': 60, 'Tier 2': 180, 'Tier 3': 365}
as_of = pd.Timestamp.now(tz='UTC').tz_localize(None).normalize()

crm['days_since_engagement'] = (as_of - crm['last_engagement_date']).dt.days
crm['target_cadence_days'] = crm['tier'].map(CADENCE_DAYS).fillna(180)
crm['review_threshold_days'] = crm['tier'].map(REVIEW_THRESHOLD_DAYS).fillna(365)
crm['engagement_urgency'] = np.where(
    crm['days_since_engagement'].isna(),
    1.0,
    (crm['days_since_engagement'] / crm['target_cadence_days']).clip(0, 2) / 2,
)
crm['engagement_status'] = np.select(
    [
        crm['last_engagement_date'].isna(),
        crm['days_since_engagement'] <= crm['target_cadence_days'],
        crm['days_since_engagement'] <= crm['review_threshold_days'],
    ],
    ['No Task', 'On Track', 'For Review'],
    default='Overdue',
)
crm['outreach_priority_score'] = 100 * (
    0.25 * crm['seniority'].map(LEVEL_SCORE).fillna(0)
    + 0.25 * crm['tier'].map(TIER_SCORE).fillna(0)
    + 0.20 * crm['relationship_quality'].map(LEVEL_SCORE).fillna(0)
    + 0.30 * crm['engagement_urgency']
)
outreach_priority = crm[[
    'contact_id', 'contact', 'company', 'tier', 'seniority',
    'relationship_quality', 'last_engagement_date',
    'days_since_engagement', 'engagement_status', 'outreach_priority_score',
]].sort_values('outreach_priority_score', ascending=False)

## 2. Build one search per unique interest

Normalization is case-insensitive and whitespace-insensitive. If ten contacts have `Golf`, the search plan contains one `golf` key and a `contact_count` of ten. The initial search—and any quality-repair search needed for duplicate URLs—is shared rather than repeated for each contact. The selected articles are later joined back to every matching contact.

In [ ]:
def interest_labels(text: str) -> list[str]:
    labels = []
    for part in clean_text(text).split(';'):
        label = re.split(r'\s+[—–-]\s+', part.strip(), maxsplit=1)[0].strip()
        if label:
            labels.append(label)
    return labels


def normalize_interest(value: str) -> str:
    return re.sub(r'\s+', ' ', clean_text(value).casefold()).strip()


membership_rows = []
for row in crm.itertuples(index=False):
    for category in ('professional', 'personal'):
        for interest in interest_labels(getattr(row, f'{category}_interests')):
            membership_rows.append({
                'contact_id': row.contact_id,
                'contact': row.contact,
                'company': row.company,
                'category': category,
                'interest': interest,
                'interest_key': normalize_interest(interest),
            })

interest_memberships = pd.DataFrame(membership_rows).drop_duplicates(
    ['contact_id', 'category', 'interest_key']
)
interest_searches = (
    interest_memberships.groupby('interest_key', as_index=False)
    .agg(
        interest=('interest', 'first'),
        contact_count=('contact_id', 'nunique'),
        membership_count=('contact_id', 'size'),
    )
    .sort_values('interest_key')
    .reset_index(drop=True)
)
interest_searches['search_query'] = interest_searches['interest'].map(
    lambda interest: f'{interest} latest news, events, and developments'
)
interest_searches['planned_search_calls'] = 1

assert interest_searches['interest_key'].is_unique
print(
    f'{len(interest_memberships)} contact-interest memberships -> '
    f'{len(interest_searches)} unique GPT web-search calls'
)
display(interest_searches.sort_values('contact_count', ascending=False).head(15))

## 3. Retrieve ten GPT web-search results per unique interest

Set `PORTKEY_LLM_MODEL` in `.env` before running this section. Each interest/date-window/model response is cached independently using the topic name plus a collision-safe hash. The background response ID is saved immediately, so a timeout or interrupted run can resume the same request without starting another search. Set `REFRESH_NEWS = True` only when fresh calls are intentional.

In [ ]:
TRACKING_PARAMS = {'fbclid', 'gclid', 'mc_cid', 'mc_eid'}


def canonical_url(url: str) -> str:
    try:
        parsed = urlparse(url)
        query = [
            (key, value)
            for key, value in parse_qsl(parsed.query, keep_blank_values=True)
            if not key.lower().startswith('utm_')
            and key.lower() not in TRACKING_PARAMS
        ]
        return urlunparse((
            parsed.scheme.lower(), parsed.netloc.lower(),
            parsed.path.rstrip('/'), '', urlencode(query), '',
        ))
    except Exception:
        return clean_text(url)


def write_json_atomic(path: Path, value) -> None:
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2))
    temporary.replace(path)


def safe_topic_slug(topic_name: str) -> str:
    slug = re.sub(r'[^a-z0-9]+', '-', clean_text(topic_name).casefold())
    return slug.strip('-')[:60] or 'untitled-topic'


def search_cache_path(
    interest_key: str, topic_name: str, query: str, start: str, end: str
) -> Path:
    payload = (
        f'{get_llm_model()}\n{SEARCH_RESULTS_PER_INTEREST}\n'
        f'{interest_key}\n{query}\n{start}\n{end}'
    )
    digest = hashlib.sha256(payload.encode()).hexdigest()[:16]
    topic_slug = safe_topic_slug(topic_name)
    return CACHE_DIR / f'web_search_{topic_slug}_{digest}.json'


def unpack_search_payload(payload) -> tuple[list[dict], dict | None]:
    if isinstance(payload, list):
        return payload, None  # Legacy cache created before usage capture.
    return payload.get('articles', []), payload.get('usage')


async def cached_news_search(
    cache_path: Path, query: str, start: str, end: str, refresh: bool
) -> tuple[dict | list, str]:
    response_id = None
    if cache_path.exists() and not refresh:
        cached_payload = json.loads(cache_path.read_text())
        if isinstance(cached_payload, dict):
            response_id = cached_payload.get('pending_response_id')
        if not response_id:
            return cached_payload, 'cache hit'
        print(f'  resuming background response {response_id}')

    def remember_pending(created_response_id: str) -> None:
        write_json_atomic(cache_path, {
            'status': 'pending',
            'pending_response_id': created_response_id,
        })

    try:
        payload = await search_news_async(
            query, start_date=start, end_date=end, include_usage=True,
            response_id=response_id,
            on_response_created=remember_pending,
        )
    except WebSearchTimeoutError as exc:
        if exc.response_id:
            remember_pending(exc.response_id)
        raise TimeoutError(
            f'{exc} Its response ID is saved in {cache_path.name}; '
            'rerun this cell to resume it without starting a new search.'
        ) from exc

    write_json_atomic(cache_path, payload)
    return payload, 'resumed call' if response_id else 'new call'


def estimated_search_cost_usd(usage: dict) -> float:
    input_tokens = int(usage.get('input_tokens', 0))
    cached_tokens = int(usage.get('cached_input_tokens', 0))
    cache_write_tokens = int(usage.get('cache_write_tokens', 0))
    regular_input_tokens = max(
        0, input_tokens - cached_tokens - cache_write_tokens
    )
    return (
        regular_input_tokens * LUNA_INPUT_USD_PER_MILLION
        + cached_tokens * LUNA_CACHED_INPUT_USD_PER_MILLION
        + cache_write_tokens * LUNA_CACHE_WRITE_USD_PER_MILLION
        + int(usage.get('output_tokens', 0)) * LUNA_OUTPUT_USD_PER_MILLION
    ) / 1_000_000 + int(usage.get('web_search_calls', 0)) * WEB_SEARCH_USD_PER_CALL


async def retrieve_news(
    search_plan: pd.DataFrame, refresh: bool = False
) -> pd.DataFrame:
    end = pd.Timestamp.now(tz='UTC').date()
    start = end - pd.Timedelta(days=LOOKBACK_DAYS)
    start_s, end_s = start.isoformat(), end.isoformat()
    records = []

    for position, row in enumerate(search_plan.itertuples(index=False), start=1):
        cache_path = search_cache_path(
            row.interest_key, row.interest, row.search_query, start_s, end_s
        )
        if not cache_path.exists() or refresh:
            print(f'[{position}/{len(search_plan)}] {row.interest}: searching...')
        search_payload, cache_status = await cached_news_search(
            cache_path, row.search_query, start_s, end_s, refresh
        )
        articles, initial_usage = unpack_search_payload(search_payload)
        usage_records = [initial_usage] if initial_usage else []

        if len(articles) < SEARCH_RESULTS_PER_INTEREST:
            print(
                f'  received {len(articles)} complete records out of '
                f'{SEARCH_RESULTS_PER_INTEREST}; repair will fill the gap'
            )
        seen_urls = set()
        unique_articles = []
        duplicates_dropped = 0

        def add_unique(raw_articles: list[dict]) -> None:
            nonlocal duplicates_dropped
            for article in raw_articles:
                url = canonical_url(clean_text(article['url']))
                if not url or url in seen_urls:
                    duplicates_dropped += 1
                    continue
                seen_urls.add(url)
                title = clean_text(article['title'])
                summary = clean_text(article['summary'])
                unique_articles.append({
                    'title': title, 'summary': summary, 'url': url,
                    'article_text': f'Title: {title}. Summary: {summary}',
                })

        add_unique(articles)
        search_passes = 1
        raw_results_received = len(articles)
        if len(unique_articles) < SEARCH_RESULTS_PER_INTEREST:
            exclusions = ' | '.join(sorted(seen_urls))
            repair_query = (
                f'{row.search_query}. Find additional results using direct '
                'article URLs only. Do not return publication homepages, '
                'topic pages, category pages, tag pages, or search pages. '
                f'Exclude every URL in this list: {exclusions}'
            )
            repair_cache_path = search_cache_path(
                row.interest_key, row.interest, repair_query, start_s, end_s
            )
            if not repair_cache_path.exists() or refresh:
                print(
                    f'  only {len(unique_articles)} unique direct URLs; '
                    'running one cached repair search...'
                )
            repair_payload, repair_status = await cached_news_search(
                repair_cache_path, repair_query, start_s, end_s, refresh
            )
            repair_status = f'repair {repair_status}'
            repair_articles, repair_usage = unpack_search_payload(repair_payload)
            if repair_usage:
                usage_records.append(repair_usage)
            search_passes += 1
            raw_results_received += len(repair_articles)
            add_unique(repair_articles)
            unique_articles = unique_articles[:SEARCH_RESULTS_PER_INTEREST]
            print(f'  {repair_status}')

        usage_totals = {
            field: sum(int(usage.get(field, 0)) for usage in usage_records)
            for field in (
                'input_tokens', 'cached_input_tokens', 'cache_write_tokens',
                'output_tokens', 'total_tokens', 'web_search_calls',
            )
        }
        estimated_cost = sum(
            estimated_search_cost_usd(usage) for usage in usage_records
        )
        cost_coverage = (
            'complete' if len(usage_records) == search_passes
            else 'partial' if usage_records
            else 'unavailable (legacy cache)'
        )

        unique_count = len(unique_articles)
        if unique_count == 0:
            print('  warning: no usable articles; skipping this interest')
            continue
        if unique_count < TOP_RESULTS_PER_INTEREST:
            print(
                f'  warning: only {unique_count} unique direct URLs are '
                f'available; selecting fewer than {TOP_RESULTS_PER_INTEREST}'
            )
        if duplicates_dropped:
            print(
                f'  dropped {duplicates_dropped} duplicate URL(s); '
                f'{unique_count} unique articles remain'
            )
        for result_number, article in enumerate(unique_articles, start=1):
            records.append({
                'interest_key': row.interest_key,
                'interest': row.interest,
                'search_query': row.search_query,
                'result_number': result_number,
                **article,
                'search_passes': search_passes,
                'raw_results_received': raw_results_received,
                'duplicates_dropped': duplicates_dropped,
                **usage_totals,
                'estimated_cost_usd': estimated_cost,
                'cost_coverage': cost_coverage,
                'pricing_as_of': PRICING_AS_OF,
            })
        print(f'[{position}/{len(search_plan)}] {row.interest}: {cache_status}')

    frame = pd.DataFrame(records)
    if frame.empty:
        raise RuntimeError('No usable articles were returned for any interest.')
    counts = frame.groupby('interest_key').size()
    missing = sorted(set(search_plan['interest_key']) - set(counts.index))
    if missing:
        print(f'Warning: no usable articles for {len(missing)} interest(s): {missing}')
    assert counts.between(1, SEARCH_RESULTS_PER_INTEREST).all(), counts.to_dict()
    return frame


articles = await retrieve_news(interest_searches, refresh=REFRESH_NEWS)
print(f'{len(articles)} articles across {articles.interest_key.nunique()} interests')
display(articles[['interest', 'result_number', 'title', 'url']].head(14))

## 4. Embed article summaries and interest labels

The exact `Title + Summary` text and each unique interest label are embedded once and cached. CRM operational fields never enter the embedding request.

In [ ]:
EMBED_CACHE_PATH = CACHE_DIR / 'embeddings.json'


def load_json_dict(path: Path) -> dict:
    return json.loads(path.read_text()) if path.exists() else {}


def embedding_key(text: str) -> str:
    return hashlib.sha256(f'{EMBEDDING_MODEL}\n{text}'.encode()).hexdigest()


async def embed_texts(texts: list[str], batch_size: int = 100) -> np.ndarray:
    texts = [clean_text(text) for text in texts]
    cache = load_json_dict(EMBED_CACHE_PATH)
    missing = list(dict.fromkeys(
        text for text in texts if embedding_key(text) not in cache
    ))
    if missing:
        client = EmbeddingClient(model=EMBEDDING_MODEL, batch_size=batch_size)
        vectors = await client.embed_texts_async(missing, normalize=True)
        for text, vector in zip(missing, vectors):
            cache[embedding_key(text)] = vector.tolist()
        write_json_atomic(EMBED_CACHE_PATH, cache)
    return np.asarray(
        [cache[embedding_key(text)] for text in texts], dtype=np.float32
    )


article_embeddings = await embed_texts(articles['article_text'].tolist())
interest_labels_ordered = interest_searches['interest'].tolist()
interest_embeddings = await embed_texts(interest_labels_ordered)
interest_embedding_lookup = dict(zip(
    interest_searches['interest_key'], interest_embeddings
))
print('Embedding matrices:', article_embeddings.shape, interest_embeddings.shape)

## 5. Rank the ten articles with BM25 + semantic similarity

BM25 is calculated independently inside each interest's ten-result pool. Its score and cosine similarity are normalized and combined transparently. The top seven are selected without any reranker.

In [ ]:
STOPWORDS = {
    'the', 'and', 'for', 'with', 'that', 'this', 'from', 'into', 'their',
    'about', 'interested', 'interest', 'interests', 'latest', 'news',
    'events', 'developments',
}


def token_list(text: str) -> list[str]:
    return [
        token
        for token in re.findall(r'(?u)\b[\w-]{3,}\b', clean_text(text).lower())
        if token not in STOPWORDS and not token.isdigit()
    ]


def bm25_scores(query_text: str, documents: list[str]) -> np.ndarray:
    document_tokens = [token_list(document) for document in documents]
    frequencies = [Counter(tokens) for tokens in document_tokens]
    lengths = np.asarray([len(tokens) for tokens in document_tokens], dtype=float)
    average_length = float(lengths.mean()) if len(lengths) else 0.0
    document_frequency = Counter()
    for tokens in document_tokens:
        document_frequency.update(set(tokens))

    scores = np.zeros(len(documents), dtype=float)
    if not documents or average_length == 0:
        return scores
    for term in token_list(query_text):
        frequency = document_frequency.get(term, 0)
        if frequency == 0:
            continue
        inverse_document_frequency = math.log(
            1 + (len(documents) - frequency + 0.5) / (frequency + 0.5)
        )
        for index, term_frequency in enumerate(frequencies):
            count = term_frequency.get(term, 0)
            if count == 0:
                continue
            length_factor = 1 - BM25_B + BM25_B * lengths[index] / average_length
            scores[index] += (
                inverse_document_frequency * count * (BM25_K1 + 1)
                / (count + BM25_K1 * length_factor)
            )
    return scores


def normalize_nonnegative(values: np.ndarray) -> np.ndarray:
    maximum = float(np.max(values)) if len(values) else 0.0
    return values / maximum if maximum > 0 else np.zeros_like(values, dtype=float)


scored_groups = []
for interest_key, group in articles.groupby('interest_key', sort=False):
    group = group.copy()
    indices = group.index.to_numpy()
    lexical = bm25_scores(group.iloc[0]['interest'], group['article_text'].tolist())
    semantic = cosine_similarity(
        np.asarray([interest_embedding_lookup[interest_key]]),
        article_embeddings[indices],
    )[0]
    group['bm25_raw_score'] = lexical
    group['bm25_score'] = normalize_nonnegative(lexical)
    group['semantic_similarity'] = semantic
    group['semantic_score'] = normalize_nonnegative(np.clip(semantic, 0, None))
    group['hybrid_score'] = (
        BM25_WEIGHT * group['bm25_score']
        + SEMANTIC_WEIGHT * group['semantic_score']
    )
    group = group.sort_values(
        ['hybrid_score', 'semantic_similarity', 'result_number'],
        ascending=[False, False, True],
    )
    group['rank'] = range(1, len(group) + 1)
    scored_groups.append(group)

scored_articles = pd.concat(scored_groups, ignore_index=True)
top_articles = scored_articles[
    scored_articles['rank'] <= TOP_RESULTS_PER_INTEREST
].copy()
counts = top_articles.groupby('interest_key').size()
available_counts = scored_articles.groupby('interest_key').size()
expected_counts = available_counts.clip(upper=TOP_RESULTS_PER_INTEREST)
assert counts.eq(expected_counts).all(), counts.to_dict()
display(top_articles[[
    'interest', 'rank', 'title', 'bm25_score',
    'semantic_similarity', 'hybrid_score',
]].head(12))

## 6. Reuse each interest's top seven for matching contacts and export Excel

The search and ranking happen once per unique interest. This final join expands those shared results back to contacts for consumption.

In [ ]:
recommendations = interest_memberships.merge(
    top_articles[[
        'interest_key', 'rank', 'title', 'summary', 'url',
        'bm25_score', 'semantic_similarity', 'hybrid_score',
    ]],
    on='interest_key', how='inner', validate='many_to_many',
)
recommendations = recommendations[[
    'contact_id', 'contact', 'company', 'category', 'interest', 'rank',
    'title', 'summary', 'url', 'bm25_score',
    'semantic_similarity', 'hybrid_score',
]].sort_values(['contact', 'category', 'interest', 'rank'])

selected_counts = top_articles.groupby('interest_key').size()
expected_rows = int(
    interest_memberships['interest_key'].map(selected_counts).sum()
)
assert len(recommendations) == expected_rows

search_audit = interest_searches[[
    'interest_key', 'interest', 'search_query', 'contact_count',
    'membership_count', 'planned_search_calls',
]].copy()
unique_article_counts = scored_articles.groupby('interest_key').size()
search_pass_counts = scored_articles.groupby('interest_key')['search_passes'].max()
raw_result_counts = scored_articles.groupby('interest_key')['raw_results_received'].max()
duplicate_counts = scored_articles.groupby('interest_key')['duplicates_dropped'].max()
search_audit['requested_articles'] = SEARCH_RESULTS_PER_INTEREST
search_audit['unique_articles_returned'] = (
    search_audit['interest_key'].map(unique_article_counts).fillna(0).astype(int)
)
search_audit['search_passes'] = search_audit['interest_key'].map(search_pass_counts).fillna(0).astype(int)
search_audit['raw_results_received'] = search_audit['interest_key'].map(raw_result_counts).fillna(0).astype(int)
search_audit['duplicate_urls_dropped'] = search_audit['interest_key'].map(duplicate_counts).fillna(0).astype(int)
search_audit['selected_articles'] = (
    search_audit['interest_key'].map(selected_counts).fillna(0).astype(int)
)
search_audit['llm_model'] = get_llm_model()
search_audit['embedding_model'] = EMBEDDING_MODEL
for metric in (
    'input_tokens', 'cached_input_tokens', 'cache_write_tokens',
    'output_tokens', 'total_tokens', 'web_search_calls',
):
    metric_by_interest = scored_articles.groupby('interest_key')[metric].max()
    search_audit[metric] = (
        search_audit['interest_key'].map(metric_by_interest).fillna(0).astype(int)
    )
estimated_cost_by_interest = scored_articles.groupby(
    'interest_key'
)["estimated_cost_usd"].max()
coverage_by_interest = scored_articles.groupby(
    'interest_key'
)["cost_coverage"].first()
search_audit['estimated_cost_usd'] = (
    search_audit['interest_key'].map(estimated_cost_by_interest).fillna(0.0)
)
search_audit['cost_coverage'] = search_audit['interest_key'].map(
    coverage_by_interest
).fillna('unavailable')
search_audit['pricing_as_of'] = PRICING_AS_OF

source_articles = scored_articles[[
    'interest', 'search_query', 'result_number', 'rank',
    'title', 'summary', 'url', 'bm25_raw_score', 'bm25_score',
    'semantic_similarity', 'hybrid_score',
]].sort_values(['interest', 'result_number'])

with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
    recommendations.to_excel(writer, sheet_name='Recommendations', index=False)
    source_articles.to_excel(writer, sheet_name='Search Results', index=False)
    search_audit.to_excel(writer, sheet_name='Unique Search Audit', index=False)
    outreach_priority.to_excel(writer, sheet_name='Outreach Priority', index=False)

print(f'Exported {len(recommendations):,} recommendation rows to {OUTPUT_XLSX}')
print(
    f'Shared search passes represented in the cache: '
    f'{int(search_audit.search_passes.sum())}; '
    f'contact-interest memberships served: {len(interest_memberships)}'
)
print(
    'Known GPT web-search estimate: '
    f"${search_audit['estimated_cost_usd'].sum():.6f} USD; "
    f"complete coverage for {search_audit['cost_coverage'].eq('complete').sum()}/"
    f'{len(search_audit)} interests'
)
display(recommendations.head(12))

## Reading the output

- **Recommendations**: seven ranked articles for every contact/category/interest membership.
- **Search Results**: the unique searched articles and their retrieval scores; the audit sheet records any reduction from ten caused by duplicate canonical URLs.
- **Unique Search Audit**: records contact reuse, token usage, web-search calls, and a local USD estimate based on the official rates dated in `pricing_as_of`. Legacy caches created before usage capture are marked partial or unavailable rather than assigned a misleading cost.
- **Outreach Priority**: the unchanged operational priority score, kept separate from article relevance.

No Tavily or Cohere call is made anywhere in this notebook.